# CMS Hospital Data Loading and Merge Checks

This notebook converts the current `data_loading.py` workflow into discrete, iterative notebook cells.

It covers:
- loading the core CMS files
- checking merge keys
- defining the HAC target
- building a general-info base table
- reshaping HCAHPS to wide format
- reshaping numeric Timely measures to wide format
- merging and saving interim datasets

## 1. Setup imports and paths

This cell makes the notebook work from inside the `notebooks/` folder by adding `src/` to the Python path.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

NOTEBOOK_DIR = Path.cwd()

# Assumes notebook is stored in repo_root/notebooks/
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from config import HAI_FILE, HCAHPS_FILE, TIMELY_FILE, GENERAL_FILE, HAC_FILE, INTERIM_DIR

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("INTERIM_DIR:", INTERIM_DIR)

## 2. Load the core CMS tables

In [ ]:
hai = pd.read_csv(HAI_FILE, dtype={"Facility ID": "string"})
hcahps = pd.read_csv(HCAHPS_FILE, dtype={"Facility ID": "string"}, low_memory=False)
timely = pd.read_csv(TIMELY_FILE, dtype={"Facility ID": "string"}, low_memory=False)
general = pd.read_csv(GENERAL_FILE, dtype={"Facility ID": "string"})
hac = pd.read_csv(HAC_FILE, dtype={"Facility ID": "string"})

## 3. Check basic shapes

In [ ]:
print("HAI shape:", hai.shape)
print("HCAHPS shape:", hcahps.shape)
print("Timely shape:", timely.shape)
print("General shape:", general.shape)
print("HAC shape:", hac.shape)

## 4. Confirm `Facility ID` exists in all five source tables

In [ ]:
print("HAI has Facility ID:", "Facility ID" in hai.columns)
print("HCAHPS has Facility ID:", "Facility ID" in hcahps.columns)
print("Timely has Facility ID:", "Facility ID" in timely.columns)
print("General has Facility ID:", "Facility ID" in general.columns)
print("HAC has Facility ID:", "Facility ID" in hac.columns)

## 5. Count unique hospitals per source table

In [ ]:
print("HAI unique Facility IDs:", hai["Facility ID"].nunique())
print("HCAHPS unique Facility IDs:", hcahps["Facility ID"].nunique())
print("Timely unique Facility IDs:", timely["Facility ID"].nunique())
print("General unique Facility IDs:", general["Facility ID"].nunique())
print("HAC unique Facility IDs:", hac["Facility ID"].nunique())

## 6. Inspect HAC columns

In [ ]:
hac.columns.tolist()

## 7. Check the raw `Payment Reduction` target distribution

In [ ]:
hac["Payment Reduction"].value_counts(dropna=False)

## 8. Build a clean binary HAC target

In [ ]:
hac_target = hac[["Facility ID", "Payment Reduction"]].copy()
hac_target = hac_target.dropna(subset=["Payment Reduction"])
hac_target["target"] = hac_target["Payment Reduction"].map({"No": 0, "Yes": 1})

print("Target shape:", hac_target.shape)
print()
print(hac_target["target"].value_counts(dropna=False))
print()
print("Target unique Facility IDs:", hac_target["Facility ID"].nunique())
print("Duplicate Facility IDs in target:", hac_target["Facility ID"].duplicated().sum())

## 9. Merge the target onto general hospital information

In [ ]:
model_base = general.merge(hac_target, on="Facility ID", how="inner")

print("Model base shape:", model_base.shape)
print("Model base unique Facility IDs:", model_base["Facility ID"].nunique())

## 10. Save the first base table

In [ ]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

output_path = INTERIM_DIR / "model_base.csv"
model_base.to_csv(output_path, index=False)

print(output_path)

## 11. Inspect the base table columns

In [ ]:
model_base.columns.tolist()

## 12. HCAHPS long-format structure: rows per hospital

In [ ]:
hcahps["Facility ID"].value_counts().describe()

## 13. Inspect HCAHPS columns

In [ ]:
hcahps.columns.tolist()

## 14. Check HCAHPS measure structure

In [ ]:
print("Unique HCAHPS Measure IDs:", hcahps["HCAHPS Measure ID"].nunique())
print()
print(hcahps["HCAHPS Measure ID"].value_counts().head(20))

## 15. Check raw HCAHPS value columns

In [ ]:
print("Patient Survey Star Rating:", hcahps["Patient Survey Star Rating"].notna().sum())
print("HCAHPS Answer Percent:", hcahps["HCAHPS Answer Percent"].notna().sum())
print("HCAHPS Linear Mean Value:", hcahps["HCAHPS Linear Mean Value"].notna().sum())

## 16. Inspect sample raw HCAHPS values

In [ ]:
print("Patient Survey Star Rating")
print(hcahps["Patient Survey Star Rating"].astype(str).value_counts().head(10))
print()
print("HCAHPS Answer Percent")
print(hcahps["HCAHPS Answer Percent"].astype(str).value_counts().head(10))
print()
print("HCAHPS Linear Mean Value")
print(hcahps["HCAHPS Linear Mean Value"].astype(str).value_counts().head(10))

## 17. Convert HCAHPS value columns to numeric where possible

In [ ]:
hcahps["star_rating_num"] = pd.to_numeric(hcahps["Patient Survey Star Rating"], errors="coerce")
hcahps["answer_percent_num"] = pd.to_numeric(hcahps["HCAHPS Answer Percent"], errors="coerce")
hcahps["linear_mean_num"] = pd.to_numeric(hcahps["HCAHPS Linear Mean Value"], errors="coerce")

print("star_rating_num:", hcahps["star_rating_num"].notna().sum())
print("answer_percent_num:", hcahps["answer_percent_num"].notna().sum())
print("linear_mean_num:", hcahps["linear_mean_num"].notna().sum())

## 18. Summarize HCAHPS measures by numeric field usage

In [ ]:
hcahps_measure_summary = (
    hcahps.groupby("HCAHPS Measure ID")[["star_rating_num", "answer_percent_num", "linear_mean_num"]]
    .apply(lambda x: x.notna().sum())
)

hcahps_measure_summary.head(20)

## 19. Build one unified numeric HCAHPS value

In [ ]:
hcahps["hcahps_value"] = (
    hcahps["answer_percent_num"]
    .combine_first(hcahps["linear_mean_num"])
    .combine_first(hcahps["star_rating_num"])
)

print("Unified HCAHPS numeric values:", hcahps["hcahps_value"].notna().sum())
print(
    "Duplicate Facility ID + HCAHPS Measure ID rows:",
    hcahps.duplicated(subset=["Facility ID", "HCAHPS Measure ID"]).sum()
)

## 20. Pivot HCAHPS to one row per hospital

In [ ]:
hcahps_wide = hcahps.pivot(
    index="Facility ID",
    columns="HCAHPS Measure ID",
    values="hcahps_value"
).reset_index()

print("HCAHPS wide shape:", hcahps_wide.shape)
print("HCAHPS wide unique Facility IDs:", hcahps_wide["Facility ID"].nunique())

## 21. Merge HCAHPS into the model base

In [ ]:
model_with_hcahps = model_base.merge(hcahps_wide, on="Facility ID", how="left")

print("Model with HCAHPS shape:", model_with_hcahps.shape)
print("Model with HCAHPS unique Facility IDs:", model_with_hcahps["Facility ID"].nunique())

## 22. Check HCAHPS completeness after the merge

In [ ]:
hcahps_feature_cols = [col for col in hcahps_wide.columns if col != "Facility ID"]
model_with_hcahps["hcahps_nonnull_count"] = model_with_hcahps[hcahps_feature_cols].notna().sum(axis=1)

print(model_with_hcahps["hcahps_nonnull_count"].describe())
print()
print(model_with_hcahps["hcahps_nonnull_count"].value_counts().sort_index())
print()
print("Hospitals with zero HCAHPS features:", (model_with_hcahps["hcahps_nonnull_count"] == 0).sum())

## 23. Check target balance by HCAHPS completeness

In [ ]:
pd.crosstab(
    model_with_hcahps["hcahps_nonnull_count"],
    model_with_hcahps["target"],
    margins=True
)

## 24. Timely long-format structure: rows per hospital

In [ ]:
timely["Facility ID"].value_counts().describe()

## 25. Inspect Timely columns

In [ ]:
timely.columns.tolist()

## 26. Check Timely measure structure

In [ ]:
print("Unique Timely Measure IDs:", timely["Measure ID"].nunique())
print()
print(timely["Measure ID"].value_counts().head(20))

## 27. Inspect raw Timely score values

In [ ]:
timely["Score"].astype(str).value_counts().head(20)

## 28. Convert Timely scores to numeric where possible

In [ ]:
timely["score_num"] = pd.to_numeric(timely["Score"], errors="coerce")

print("Numeric Timely score count:", timely["score_num"].notna().sum())
print("Total Timely rows:", len(timely))

## 29. Summarize numeric Timely coverage by measure

In [ ]:
timely_measure_summary = timely.groupby("Measure ID")["score_num"].apply(lambda x: x.notna().sum())
timely_measure_summary.sort_values(ascending=False)

## 30. Keep only numeric Timely rows and pivot wide

In [ ]:
timely_numeric = timely[timely["score_num"].notna()].copy()

timely_wide = timely_numeric.pivot(
    index="Facility ID",
    columns="Measure ID",
    values="score_num"
).reset_index()

print("Timely wide shape:", timely_wide.shape)
print("Timely wide unique Facility IDs:", timely_wide["Facility ID"].nunique())

## 31. Merge Timely into the HCAHPS model table

In [ ]:
model_with_hcahps_timely = model_with_hcahps.merge(timely_wide, on="Facility ID", how="left")

print("Model with HCAHPS + Timely shape:", model_with_hcahps_timely.shape)
print("Model with HCAHPS + Timely unique Facility IDs:", model_with_hcahps_timely["Facility ID"].nunique())

## 32. Check Timely completeness after the merge

In [ ]:
timely_feature_cols = [col for col in timely_wide.columns if col != "Facility ID"]
model_with_hcahps_timely["timely_nonnull_count"] = model_with_hcahps_timely[timely_feature_cols].notna().sum(axis=1)

print(model_with_hcahps_timely["timely_nonnull_count"].describe())
print()
print(model_with_hcahps_timely["timely_nonnull_count"].value_counts().sort_index())
print()
print("Hospitals with zero Timely features:", (model_with_hcahps_timely["timely_nonnull_count"] == 0).sum())

## 33. Check target balance by Timely completeness

In [ ]:
pd.crosstab(
    model_with_hcahps_timely["timely_nonnull_count"],
    model_with_hcahps_timely["target"],
    margins=True
)

## 34. Save the merged HCAHPS + Timely file

In [ ]:
output_path_2 = INTERIM_DIR / "model_with_hcahps_timely.csv"
model_with_hcahps_timely.to_csv(output_path_2, index=False)

print(output_path_2)

## 35. Build a predictor-ready version by dropping the leakage column

In [ ]:
model_ready = model_with_hcahps_timely.drop(columns=["Payment Reduction"]).copy()

output_path_3 = INTERIM_DIR / "model_ready_v1.csv"
model_ready.to_csv(output_path_3, index=False)

print("Model-ready file shape:", model_ready.shape)
print("Saved model-ready file to:")
print(output_path_3)

## 36. Inspect final object/string columns

In [ ]:
model_ready.select_dtypes(include=["object", "string"]).columns.tolist()

## 37. Optional next step

Use the next notebook to:
- separate ID / leakage / free-text columns from real features
- convert numeric-looking text columns
- define preprocessing for categorical and missing values
- build your first baseline model